In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import copy
import gc
import glob
import math
import os
import pathlib
import time
from collections import OrderedDict
from itertools import product

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import xarray as xr
from tqdm.notebook import tqdm

plt.rcParams["font.family"] = "serif"
plt.style.use("tableau-colorblind10")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = r":4096:8"  # to make calculations deterministic

In [ ]:
from scripts.train_pdd_kolmogorov_flow import DT, initialize_trainer
from src.configs.kolmogorov_flow_config import KolmogorovFlowUnetConfig
from src.models.dynamics.surrogate_simulators import run_simulation
from src.util.random_seed_helper import set_seeds

# Define constants

In [ ]:
DEVICE = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
ROOT_DIR = pathlib.Path(os.environ["PYTHONPATH"].split(":")[0]).resolve()
print(f"{ROOT_DIR=}, {DEVICE=}")

# Plot deep-learning data

In [ ]:
path = f"{ROOT_DIR}/data/DL_data/kolmogorov_flow/kf_100x40x40x40_001.npy"
data = np.load(path)
assert data.shape == (100, 40, 40, 40)  # batch, t, y, x

In [ ]:
fig, axes = plt.subplots(5, 8, figsize=(15, 8))
for i, ax in enumerate(axes.flatten()):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    d = data[0, i].transpose()
    xs = np.arange(d.shape[0])
    ys = np.arange(d.shape[1])
    xs, ys = np.meshgrid(xs, ys, indexing="ij")
    ret = ax.pcolormesh(xs, ys, d, shading="nearest", cmap="RdBu_r")
    fig.colorbar(ret, ax=ax)
    ax.set_title(f"Time index = {i:02}")

plt.tight_layout()
plt.show()

# Load a trained PDD

In [ ]:
p = f"{ROOT_DIR}/configs/kolmogorov_flow_unet.yml"
config = KolmogorovFlowUnetConfig.load(p)
p = f"{ROOT_DIR}/data/DL_model/kolmogorov_flow/kolmogorov_flow_unet"
trainer, dataset = initialize_trainer(
    config, str(DEVICE), ROOT_DIR, result_dir=p, kind="test"
)
trainer.load_only_model(milestone=30_000)
_ = trainer.model.noise_estimate_fn.closure.eval()

# Perform simulation

In [ ]:
dict_results = {}
for diffusion_idx in [0, 100, 200]:
    ground_truth, pred = run_simulation(
        trainer=trainer,
        dataset=dataset,
        config=config,
        n_batches=5,
        diffusion_index=diffusion_idx,
        dt=DT,
        n_steps=40,
        device=DEVICE,
        is_noise_off=True,
    )
    dict_results[diffusion_idx] = {"gt": ground_truth, "pred": pred}

In [ ]:
for diffusion_idx in [0, 100, 200]:
    fig, axes = plt.subplots(2, 8, figsize=(20, 4))

    for i, j in product(range(2), range(8)):
        if i == 0:
            data = dict_results[diffusion_idx]["gt"][0, 0, :40]
            ttl = "Physics"
        elif i == 1:
            data = dict_results[diffusion_idx]["pred"][0, 0, :40]
            ttl = "Surrogate"
        assert data.shape == (40, 40, 40)  # t, y, x
        scaled = dataset.standardize(data)

        it = j * 5
        ax = axes[i, j]
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_aspect("equal")
        d = scaled[it].transpose()
        xs = np.arange(d.shape[0])
        ys = np.arange(d.shape[1])
        xs, ys = np.meshgrid(xs, ys, indexing="ij")
        ret = ax.pcolormesh(
            xs, ys, d, shading="nearest", cmap="RdBu_r", vmin=-2, vmax=2
        )
        fig.colorbar(ret, ax=ax)
        ax.set_title(f"{ttl}: it = {it:02}")

    plt.suptitle(f"diffusion scale = {float(diffusion_idx)/config.num_timesteps:.2f}")
    plt.tight_layout()
    plt.show()

# Perform generation

In [ ]:
set_seeds(0)
intermediates = trainer.model.sample(
    batch_size=5,
    corrector_snr=0.3,
    num_corrector_steps=3,
)

In [ ]:
data = intermediates[1][4].numpy()

fig, axes = plt.subplots(5, 8, figsize=(15, 8))
for i, ax in enumerate(axes.flatten()):
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    d = data[0, i].transpose()
    xs = np.arange(d.shape[0])
    ys = np.arange(d.shape[1])
    xs, ys = np.meshgrid(xs, ys, indexing="ij")
    ret = ax.pcolormesh(xs, ys, d, shading="nearest", cmap="RdBu_r")
    fig.colorbar(ret, ax=ax)
    ax.set_title(f"Time index = {i:02}")

plt.tight_layout()
plt.show()

# Perform super-resolution

In [ ]:
set_seeds(0)
_, lr = run_simulation(
    trainer=trainer,
    dataset=dataset,
    config=config,
    n_batches=5,
    diffusion_index=200,
    dt=DT,
    n_steps=80,
    device=DEVICE,
    is_noise_off=False,
)
lr = dataset.standardize(lr[:, :, -40:, :])
assert lr.shape == (5, 1, 40, 40, 40)

In [ ]:
set_seeds(0)
intermediates = trainer.model._p_loop(
    n_timesteps=201,
    n_batches=5,
    img=torch.from_numpy(lr).to(device=DEVICE, dtype=torch.float32).view(5, 1, 40, -1),
    num_corrector_steps=3,
    corrector_snr=0.3,
)

In [ ]:
sr = intermediates[1].numpy().reshape(5, 1, 40, 40, 40)

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(20, 4))

for i, j in product(range(2), range(8)):
    if i == 0:
        scaled = sr[1, 0, :40]
        ttl = "SR"
    elif i == 1:
        scaled = lr[1, 0, :40]
        ttl = "LR"
    assert scaled.shape == (40, 40, 40)  # t, y, x

    it = 10 + j
    ax = axes[i, j]
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect("equal")
    d = scaled[it].transpose()
    xs = np.arange(d.shape[0])
    ys = np.arange(d.shape[1])
    xs, ys = np.meshgrid(xs, ys, indexing="ij")
    ret = ax.pcolormesh(xs, ys, d, shading="nearest", cmap="RdBu_r", vmin=-2, vmax=2)
    fig.colorbar(ret, ax=ax)
    ax.set_title(f"{ttl}: it = {it:02}")

plt.tight_layout()
plt.show()